In [23]:
import numpy as np
import pandas as pd
from pandas import Series, DataFrame
import math
%load_ext autoreload
%autoreload 2            

import os, sys
sys.path.append(os.path.abspath(".."))   # ".." = thư mục cha của notebooks/ = algo-ml

from src import journal

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
df = pd.read_csv("../data/raw/EURUSD_H1.csv", index_col="timestamp",parse_dates=True)
df

,open,high,low,close,volume
timestamp,,,,,
2025-01-01 22:00:00+00:00,1.03503,1.03577,1.03483,1.03550,1443.22
2025-01-01 23:00:00+00:00,1.03550,1.03563,1.03462,1.03507,4447.37
2025-01-02 00:00:00+00:00,1.03508,1.03564,1.03472,1.03520,3681.37
2025-01-02 01:00:00+00:00,1.03521,1.03606,1.03442,1.03578,5012.45
2025-01-02 02:00:00+00:00,1.03577,1.03712,1.03577,1.03709,2791.47
...,...,...,...,...,...
2025-12-31 17:00:00+00:00,1.17346,1.17390,1.17312,1.17358,3662.76
2025-12-31 18:00:00+00:00,1.17357,1.17386,1.17302,1.17357,3944.61
2025-12-31 19:00:00+00:00,1.17356,1.17432,1.17356,1.17407,3321.68


In [25]:
assert len(df) == 6225
assert isinstance(df.index, pd.DatetimeIndex)
df.dtypes

open      float64
high      float64
low       float64
close     float64
volume    float64
dtype: object

In [26]:
df.head()

,open,high,low,close,volume
timestamp,,,,,
2025-01-01 22:00:00+00:00,1.03503,1.03577,1.03483,1.03550,1443.22
2025-01-01 23:00:00+00:00,1.03550,1.03563,1.03462,1.03507,4447.37
2025-01-02 00:00:00+00:00,1.03508,1.03564,1.03472,1.03520,3681.37
2025-01-02 01:00:00+00:00,1.03521,1.03606,1.03442,1.03578,5012.45
2025-01-02 02:00:00+00:00,1.03577,1.03712,1.03577,1.03709,2791.47


In [27]:
df["range"] = df["high"] - df["low"]
df["body"] = abs(df["close"] - df["open"])
df["ret"] = df["close"].pct_change()
df

,open,high,low,close,volume,range,body,ret
timestamp,,,,,,,,
2025-01-01 22:00:00+00:00,1.03503,1.03577,1.03483,1.03550,1443.22,0.00094,0.00047,NaN
2025-01-01 23:00:00+00:00,1.03550,1.03563,1.03462,1.03507,4447.37,0.00101,0.00043,-0.000415
2025-01-02 00:00:00+00:00,1.03508,1.03564,1.03472,1.03520,3681.37,0.00092,0.00012,0.000126
2025-01-02 01:00:00+00:00,1.03521,1.03606,1.03442,1.03578,5012.45,0.00164,0.00057,0.000560
2025-01-02 02:00:00+00:00,1.03577,1.03712,1.03577,1.03709,2791.47,0.00135,0.00132,0.001265
...,...,...,...,...,...,...,...,...
2025-12-31 17:00:00+00:00,1.17346,1.17390,1.17312,1.17358,3662.76,0.00078,0.00012,0.000094
2025-12-31 18:00:00+00:00,1.17357,1.17386,1.17302,1.17357,3944.61,0.00084,0.00000,-0.000009
2025-12-31 19:00:00+00:00,1.17356,1.17432,1.17356,1.17407,3321.68,0.00076,0.00051,0.000426


In [28]:
df.iloc[0]

open         1.03503
high         1.03577
low          1.03483
close        1.03550
volume    1443.22000
range        0.00094
body         0.00047
ret              NaN
Name: 2025-01-01 22:00:00+00:00, dtype: float64

In [29]:
df['ret'].isna().sum()

np.int64(1)

In [30]:
threshold = df['range'].quantile(0.99)


In [31]:
mask = df["range"] > threshold      # True/False
big = df[mask]                      # 63 nến
hours = big.index.hour

hours

Index([11, 14, 13, 13,  0, 23, 18, 15, 13, 15, 13, 14, 10, 20, 23, 10, 11, 13,
       15,  7, 10, 12, 22, 14,  7, 12, 17, 18,  7, 12, 14, 15, 19,  0,  7,  8,
       10, 12, 13,  2, 21, 22, 13, 13,  7,  8, 11, 23, 12, 13, 12,  0, 12, 15,
       12, 18, 12, 14,  7, 12, 12, 18, 18],
      dtype='int32', name='timestamp')

In [32]:
df[['range', 'body', 'ret']].describe()

,range,body,ret
count,6225.000000,6225.000000,6224.000000
mean,0.001525,0.000731,0.000021
std,0.001145,0.000842,0.001013
min,0.000200,0.000000,-0.010560
25%,0.000830,0.000220,-0.000427
50%,0.001230,0.000490,0.000009
75%,0.001840,0.000950,0.000432
max,0.016690,0.014060,0.012328


In [33]:
test = df.copy()
test[test['range'] > threshold]['body'] = 999      # cách 1: chained
print((test['body'] == 999).sum())                 # dự đoán: 0

0


C:\Users\dungt\AppData\Local\Temp\ipykernel_9876\3331125607.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/docs/dev/user_guide/migration.html#chained-assignment
  test[test['range'] > threshold]['body'] = 999      # cách 1: chained


In [34]:
test.loc[test['range'] > threshold, 'body'] = 999  # cách 2: một bước
print((test['body'] == 999).sum())     

63
